In [1]:
# Source - https://stackoverflow.com/a
# Posted by G M, modified by community. See post 'Timeline' for change history
if 'google.colab' in str(get_ipython()):
  !git clone https://github.com/Vladislavicious/jenga_ml.git
  %cd jenga_ml
  !git switch dev

  !pip install -r requirements.txt
else:
  import torch
  print(torch.__version__)
  print(torch.version.cuda)

  if torch.cuda.is_available():
      print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
  else:
      print("No GPU available. Training will run on CPU.")
  print('Not running on CoLab')

2.9.1+cpu
None
No GPU available. Training will run on CPU.
Not running on CoLab


In [2]:
import random
import os

from environment import make_jenga_env

import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env


In [3]:
random.seed(123)
np.random.seed(123)
CHECK_STEPS = 1024

In [4]:
# Начинаем с 1 блока
steps_per_stage = [200_000, 500_000]
n_blocks = len(steps_per_stage)
num_envs = 8

def make_env():
        return make_jenga_env(n_blocks=n_blocks, render=True)

In [5]:
env = make_vec_env(make_env, n_envs=num_envs, vec_env_cls=SubprocVecEnv)
env = VecNormalize(env, norm_obs=True, norm_reward=False, clip_obs=10.0)
# env = make_env()

model = PPO(
            "MlpPolicy",
            env,
            verbose=1,
            seed=123,
        )


Using cpu device


In [ ]:
for stage in range(n_blocks):
    curriculum_level = stage + 1
    print(f"Этап {curriculum_level}: управление первыми {curriculum_level} блоками")

    # Передаём уровень в каждую подсреду
    env.env_method("set_curriculum_level", curriculum_level)
    model.learn(
        total_timesteps=steps_per_stage[stage],
        reset_num_timesteps=False,
        tb_log_name=f"level_{curriculum_level}"
    )

    model.save(f"./models/curriculum_level_{curriculum_level}")

Этап 1: управление первыми 1 блоками
------------------------------
| time/              |       |
|    fps             | 1893  |
|    iterations      | 1     |
|    time_elapsed    | 8     |
|    total_timesteps | 16384 |
------------------------------


In [ ]:
single_env = make_env()
single_env.set_curriculum_level(n_blocks)

obs, _ = single_env.reset()
for i in range(CHECK_STEPS * 2):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = single_env.step(action)
    single_env.render()
    if terminated or truncated:
        obs, _ = single_env.reset()
    single_env.env.debug_output()
    if i == CHECK_STEPS - 2:
        print("hi")